In [1]:
import subprocess, sys

# 버전 고정 없는 패키지: 없을 때만 설치
_basic = ['pandas', 'numpy', 'scikit-learn', 'catboost', 'plotly', 'pyarrow']
_import_map = {'scikit-learn': 'sklearn', 'pydantic-core': 'pydantic_core'}
for _pkg in _basic:
    _name = _import_map.get(_pkg, _pkg.replace('-', '_'))
    try:
        __import__(_name)
    except ImportError:
        print(f'{_pkg} 설치 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', _pkg, '-q'])
        print(f'{_pkg} 설치 완료')

# gradio: huggingface_hub 버전에 맞게 최신으로 유지
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gradio', '--upgrade', '-q'])
print('✅ 패키지 확인 완료')

✅ 패키지 확인 완료


# GNN 기반 사기 리뷰 탐지 — [07] 대화형 대시보드

**목적**: 리뷰 텍스트와 별점을 입력받아 사기 확률을 즉시 예측하는 대화형 대시보드

| 항목 | 내용 |
|------|------|
| **입력** | 리뷰 텍스트, 별점, (선택) 유저 ID / 식당 ID |
| **출력** | 사기 확률 게이지, SHAP 피처 기여도 차트, 키워드 탐지 결과 |
| **모델** | CatBoost — 텍스트·행동 피처 82차원 |
| **비고** | 최종 모델(06, CARE-HAN+CatBoost 206차원) PR-AUC **0.9279** 대비 GNN 구조 신호 미포함 |

---

## 노트북 구성

| 섹션 | 내용 |
|------|------|
| [01] | 환경 설정 및 데이터 로드 |
| [02] | 피처 엔지니어링 (텍스트 수치 피처 + TF-IDF SVD) |
| [03] | 82차원 피처 행렬 조립 |
| [04] | CatBoost 학습 및 평가 |
| [05] | 조회 테이블 구축 및 전체 사기 점수 사전 계산 |
| [06] | 대시보드 실행 (Gradio) |


## [01] 환경 설정 및 데이터 로드

In [2]:
import os, warnings, pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, f1_score, classification_report
try:
    from catboost import CatBoostClassifier, Pool
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("⚠️ catboost 미설치 — pip install catboost")

try:
    BASE_DIR = str(Path(__vsc_ipynb_file__).parent)
except NameError:
    BASE_DIR = str(Path().resolve())
os.chdir(BASE_DIR)

# ── 키워드 사전 (03_1 분석 결과 기반) ──────────────────────────────
# 사기 리뷰 특징어: 이야기 구조, 대인 관계 표현
FRAUD_KW  = {'told','said','asked','manager','table','owner','call',
             'rude','experience','server','waitress','waiter',
             'horrible','worst','complained','never','disgusting','terrible','awful'}
# 정상 리뷰 특징어: 음식 맛/식감 묘사
NORMAL_KW = {'topped','texture','creamy','sweetness','pieces','rich',
             'tender','crispy','fluffy','flavor','delicious','fresh',
             'tasty','amazing','fantastic','loved','enjoyed'}
# 메뉴 관련 단어
MENU_KW   = {'food','steak','burger','pizza','pasta','chicken','fish',
             'salad','sandwich','soup','noodle','rice','sauce','cheese',
             'beef','pork','bread','dessert','cake','coffee','drink'}
# 감성 단어
POS_WORDS = {'good','great','excellent','amazing','wonderful','love','best',
             'perfect','fantastic','delicious','fresh','friendly','enjoyed','recommend','awesome'}
NEG_WORDS = {'bad','terrible','awful','horrible','worst','rude','slow',
             'dirty','disgusting','disappointing','never','poor'}

CFG = {
    'tfidf_max_features': 5000,
    'svd_n_components'  : 64,
    'random_state'      : 42,
    'cutoff_date'       : '2014-06-28',   # 04/06 노트북과 동일한 시간 기반 분할 기준일
}

print('환경 설정 완료')
print(f'작업 디렉토리: {BASE_DIR}')


환경 설정 완료
작업 디렉토리: /Users/choseoyoung/Desktop/GNN 학술제


In [3]:
# 기본 샘플 데이터 로드
df = pd.read_parquet('yelpzip_sampled.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 별점 파생 피처 (index 정렬된 csv — 행 순서 = yelpzip_sampled 행 순서)
feat_rating = pd.read_csv('features_rating.csv', index_col=0).reset_index(drop=True)

# 버스트 피처 (유저 단위)
feat_burst = pd.read_csv('burst_features.csv')

N = len(df)
print(f'데이터: {N:,}건  |  사기: {df["label"].sum():,} ({df["label"].mean()*100:.1f}%)  |  정상: {(df["label"]==0).sum():,}')
print(f'기간  : {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'유저  : {df["user_id"].nunique():,}명  |  식당: {df["prod_id"].nunique():,}개')


데이터: 50,950건  |  사기: 12,659 (24.8%)  |  정상: 38,291
기간  : 2013-01-01 ~ 2014-12-31
유저  : 17,439명  |  식당: 200개


## [02] 피처 엔지니어링

In [4]:
def extract_text_stats(text: str) -> dict:
    """
    리뷰 텍스트에서 수치 피처 10개를 추출합니다.
    학습 시와 새 리뷰 추론 시 동일한 함수를 사용해 일관성을 보장합니다.
    """
    text  = str(text)
    words = text.lower().split()
    nw    = max(len(words), 1)
    nc    = max(len(text),  1)

    def cnt(kw_set):
        return sum(1 for w in words if w in kw_set)

    fi = cnt(FRAUD_KW)  / nw
    ni = cnt(NORMAL_KW) / nw
    return {
        'text_len'        : len(text),
        'word_count'      : nw,
        'digit_ratio'     : sum(c.isdigit() for c in text) / nc,
        'special_ratio'   : sum(1 for c in text if not c.isalnum() and not c.isspace()) / nc,
        'ttr'             : len(set(words)) / nw,
        'sentiment'       : (cnt(POS_WORDS) - cnt(NEG_WORDS)) / nw,
        'menu_mentions'   : cnt(MENU_KW),
        'fraud_intensity' : fi,
        'normal_intensity': ni,
        'vocab_discrim'   : fi - ni,
    }

# 전체 데이터에 일괄 적용
print('텍스트 피처 추출 중...')
text_stats = df['text'].apply(extract_text_stats).apply(pd.Series)
print(f'완료: {text_stats.shape}  |  피처: {text_stats.columns.tolist()}')


텍스트 피처 추출 중...
완료: (50950, 10)  |  피처: ['text_len', 'word_count', 'digit_ratio', 'special_ratio', 'ttr', 'sentiment', 'menu_mentions', 'fraud_intensity', 'normal_intensity', 'vocab_discrim']


In [5]:
# ── 시간 기반 Train / Test 분할 ──────────────────────────────────────
# 04/06 노트북과 동일한 cutoff 사용 → 모델 비교 일관성 확보
cutoff     = pd.Timestamp(CFG['cutoff_date'])
train_mask = (df['date'] <= cutoff).values
test_mask  = ~train_mask
train_idx  = np.where(train_mask)[0]
test_idx   = np.where(test_mask)[0]

print(f'Train: {len(train_idx):,}건 ({train_mask.mean()*100:.1f}%)  사기율: {df["label"].iloc[train_idx].mean():.3f}')
print(f'Test : {len(test_idx):,}건 ({test_mask.mean()*100:.1f}%)  사기율: {df["label"].iloc[test_idx].mean():.3f}')

# ── TF-IDF + SVD (Train 기준 fit — 데이터 누수 방지) ───────────────
print('\nTF-IDF + SVD 학습 중...')
tfidf = TfidfVectorizer(
    max_features=CFG['tfidf_max_features'],
    ngram_range=(1, 2), min_df=3, max_df=0.95,
    strip_accents='unicode', sublinear_tf=True,
)
tfidf.fit(df['text'].iloc[train_idx].astype(str))
tfidf_mat = tfidf.transform(df['text'].astype(str))

svd = TruncatedSVD(n_components=CFG['svd_n_components'], random_state=CFG['random_state'])
svd.fit(tfidf_mat[train_idx])
svd_mat = svd.transform(tfidf_mat).astype(np.float32)

print(f'완료 | 설명 분산: {svd.explained_variance_ratio_.sum():.3f}')


Train: 36,774건 (72.2%)  사기율: 0.235
Test : 14,176건 (27.8%)  사기율: 0.282

TF-IDF + SVD 학습 중...
완료 | 설명 분산: 0.127


## [03] 82차원 피처 행렬 조립

In [6]:
# 시간 피처 추가
df['weekday'] = df['date'].dt.weekday
df['month']   = df['date'].dt.month

# features_rating.csv 결합 (index 기반 — 행 순서 일치)
df = pd.concat([
    df.reset_index(drop=True),
    feat_rating[['rating_deviation', 'is_cold_start', 'is_single_review']].reset_index(drop=True),
], axis=1)
df[['rating_deviation', 'is_cold_start', 'is_single_review']] = \
    df[['rating_deviation', 'is_cold_start', 'is_single_review']].fillna(0)

# burst_features.csv 결합 (user_id 기준)
df = df.merge(feat_burst[['user_id', 'user_burst_max7d', 'user_burst_ratio']],
              on='user_id', how='left')
df[['user_burst_max7d', 'user_burst_ratio']] = \
    df[['user_burst_max7d', 'user_burst_ratio']].fillna(0)

# 텍스트 수치 피처 결합
df = pd.concat([df.reset_index(drop=True), text_stats.reset_index(drop=True)], axis=1)

# ── 피처 컬럼 순서 정의 (추론 시에도 반드시 동일 순서 유지) ────────
FEAT_COLS = [
    'rating', 'weekday', 'month',
    'rating_deviation', 'is_cold_start', 'is_single_review',
    'user_burst_max7d', 'user_burst_ratio',
    'text_len', 'word_count', 'digit_ratio', 'special_ratio',
    'ttr', 'sentiment', 'menu_mentions',
    'fraud_intensity', 'normal_intensity', 'vocab_discrim',
]
SVD_COLS  = [f'SVD_{i}' for i in range(CFG['svd_n_components'])]
ALL_FEATS = FEAT_COLS + SVD_COLS          # 18 + 64 = 82차원

# ── 스케일러 (Train 기준 fit) ────────────────────────────────────────
num_mat = df[FEAT_COLS].values.astype(np.float32)
scaler  = StandardScaler()
scaler.fit(num_mat[train_idx])
num_scaled = scaler.transform(num_mat)

X = np.hstack([num_scaled, svd_mat]).astype(np.float32)
y = df['label'].values

print(f'피처 행렬: {X.shape}  (수치 {len(FEAT_COLS)}개 + SVD {len(SVD_COLS)}차원 = {X.shape[1]}차원)')
print(f'NaN 포함 여부: {np.isnan(X).sum()}개')


피처 행렬: (50950, 82)  (수치 18개 + SVD 64차원 = 82차원)
NaN 포함 여부: 0개


## [04] CatBoost 학습 및 평가

In [7]:
if not CATBOOST_AVAILABLE:
    print("⚠️ catboost 미설치 — 이 셀을 건너뜁니다.")
    print("   VS Code 터미널에서: pip install catboost")
    model = None
else:
    MODEL_PATH = 'best_catboost_82.cbm'

    if os.path.exists(MODEL_PATH):
        model = CatBoostClassifier()
        model.load_model(MODEL_PATH)
        print(f'저장된 모델 로드: {MODEL_PATH}')
    else:
        X_tr, y_tr = X[train_idx], y[train_idx]

        val_cut    = int(len(X_tr) * 0.8)
        X_tr2, X_val = X_tr[:val_cut], X_tr[val_cut:]
        y_tr2, y_val = y_tr[:val_cut], y_tr[val_cut:]

        model = CatBoostClassifier(
            iterations=1000, depth=6, learning_rate=0.05,
            loss_function='Logloss', eval_metric='AUC',
            class_weights={0: 1, 1: 3},
            early_stopping_rounds=50, verbose=200,
            random_seed=CFG['random_state'],
        )
        model.fit(Pool(X_tr2, y_tr2), eval_set=Pool(X_val, y_val))
        model.save_model(MODEL_PATH)
        print(f'\n모델 저장 완료: {MODEL_PATH}')

저장된 모델 로드: best_catboost_82.cbm


In [8]:
# ── 성능 평가 ────────────────────────────────────────────────────────
probs    = model.predict_proba(X[test_idx])[:, 1]
preds    = (probs >= 0.5).astype(int)
pr_auc   = average_precision_score(y[test_idx], probs)
macro_f1 = f1_score(y[test_idx], preds, average='macro')

print('=' * 55)
print('  대시보드 모델 (CatBoost 82차원) — Test Set')
print('=' * 55)
print(f'  PR-AUC  : {pr_auc:.4f}')
print(f'  Macro F1: {macro_f1:.4f}')
print()
print('  [비교] 최종 모델 06 (CARE-HAN + CatBoost 206차원)')
print('         PR-AUC  : 0.9279  |  Macro F1: 0.9136')
print('         → GNN 구조 신호(128차원) 포함으로 더 높은 성능')
print()
print(classification_report(y[test_idx], preds,
                             target_names=['정상(0)', '사기(1)'], digits=4))

# ── 피처 중요도 시각화 (상위 20개) ──────────────────────────────────
fi_series = pd.Series(model.get_feature_importance(), index=ALL_FEATS).nlargest(20)

fig = px.bar(
    fi_series[::-1], orientation='h',
    title='CatBoost 피처 중요도 상위 20개',
    labels={'value': '중요도', 'index': '피처'},
    color=fi_series[::-1].values,
    color_continuous_scale='Blues',
)
fig.update_layout(coloraxis_showscale=False, height=520,
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()


  대시보드 모델 (CatBoost 82차원) — Test Set
  PR-AUC  : 0.6432
  Macro F1: 0.7296

  [비교] 최종 모델 06 (CARE-HAN + CatBoost 206차원)
         PR-AUC  : 0.9279  |  Macro F1: 0.9136
         → GNN 구조 신호(128차원) 포함으로 더 높은 성능

              precision    recall  f1-score   support

       정상(0)     0.8579    0.8206    0.8388     10172
       사기(1)     0.5895    0.6546    0.6204      4004

    accuracy                         0.7737     14176
   macro avg     0.7237    0.7376    0.7296     14176
weighted avg     0.7821    0.7737    0.7771     14176



## [05] 조회 테이블 구축 및 전체 사기 점수 사전 계산

In [9]:
# ── Train 데이터 기준 조회 테이블 구축 ──────────────────────────────
# (새 리뷰 추론 시 사용 — Test 데이터 정보가 섞이지 않도록 Train만 사용)
df_train = df[train_mask]

PROD_AVG_RATING   = df_train.groupby('prod_id')['rating'].mean().to_dict()
PROD_REVIEW_COUNT = df_train.groupby('prod_id').size().to_dict()
PROD_IS_COLD      = {pid: int(cnt < 5) for pid, cnt in PROD_REVIEW_COUNT.items()}

USER_BURST     = feat_burst.set_index('user_id')[['user_burst_max7d', 'user_burst_ratio']].to_dict('index')
user_rc        = df_train.groupby('user_id').size()
USER_IS_SINGLE = {uid: int(cnt <= 1) for uid, cnt in user_rc.items()}

print(f'식당 조회 테이블: {len(PROD_AVG_RATING)}개')
print(f'유저 조회 테이블: {len(USER_BURST)}명')


식당 조회 테이블: 200개
유저 조회 테이블: 782명


In [10]:
# ── 전체 50,950건 사기 점수 사전 계산 ──────────────────────────────
PRECOMP_PATH = 'precomputed_dash.parquet'

if os.path.exists(PRECOMP_PATH):
    df_precomp = pd.read_parquet(PRECOMP_PATH)
    print(f'사전 계산 결과 로드: {len(df_precomp):,}건')
else:
    all_probs = model.predict_proba(X)[:, 1]
    df_precomp = df[['user_id', 'prod_id', 'rating', 'label', 'date', 'text']].copy()
    df_precomp['fraud_score'] = all_probs
    df_precomp['fraud_pred']  = (all_probs >= 0.5).astype(int)
    df_precomp['review_idx']  = df_precomp.index
    df_precomp.to_parquet(PRECOMP_PATH)
    print(f'사전 계산 완료: {len(df_precomp):,}건 저장 → {PRECOMP_PATH}')

# ── 아티팩트 저장 (커널 재시작 없이 대시보드 재실행 가능하도록) ────
artifacts = {
    'tfidf': tfidf, 'svd': svd, 'scaler': scaler,
    'feat_cols': FEAT_COLS, 'all_feats': ALL_FEATS,
    'prod_avg_rating': PROD_AVG_RATING,
    'prod_is_cold'   : PROD_IS_COLD,
    'user_burst'     : USER_BURST,
    'user_is_single' : USER_IS_SINGLE,
}
with open('dashboard_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print('아티팩트 저장 완료: dashboard_artifacts.pkl')


사전 계산 결과 로드: 50,950건
아티팩트 저장 완료: dashboard_artifacts.pkl


## [06] 대시보드 실행 (Gradio)

In [11]:
import pickle, hashlib, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

_F = '-apple-system, BlinkMacSystemFont, sans-serif'

# ── 키워드 사전 ──────────────────────────────────────────────────────
_FRAUD_KW  = {'told','said','asked','manager','table','owner','call','rude',
              'experience','server','waitress','waiter','horrible','worst',
              'complained','never','disgusting','terrible','awful'}
_NORMAL_KW = {'topped','texture','creamy','sweetness','pieces','rich',
              'tender','crispy','fluffy','flavor','delicious','fresh',
              'tasty','amazing','fantastic','loved','enjoyed'}
_MENU_KW   = {'food','steak','burger','pizza','pasta','chicken','fish',
              'salad','sandwich','soup','noodle','rice','sauce','cheese',
              'beef','pork','bread','dessert','cake','coffee','drink'}
_POS_WORDS = {'good','great','excellent','amazing','wonderful','love','best',
              'perfect','fantastic','delicious','fresh','friendly','enjoyed','recommend','awesome'}
_NEG_WORDS = {'bad','terrible','awful','horrible','worst','rude','slow',
              'dirty','disgusting','disappointing','never','poor'}

SHAP_META = {
    'text_len'        : ('텍스트 길이',    '리뷰가 너무 짧아요'),
    'user_burst_ratio': ('버스트 집중도',  '짧은 기간에 몰아썼어요'),
    'rating_deviation': ('별점 이탈도',    '평균과 동떨어진 별점이에요'),
    'word_count'      : ('단어 수',        '내용이 너무 적어요'),
    'user_burst_max7d': ('7일내 최대 리뷰','최근 활동이 과도해요'),
    'fraud_intensity' : ('사기 어휘 강도', '사기 패턴 단어가 많아요'),
    'normal_intensity': ('정상 어휘 강도', '음식 관련 표현이 있어요'),
    'vocab_discrim'   : ('변별 어휘 점수', '어휘 패턴이 특이해요'),
    'ttr'             : ('어휘 다양성',    '단어 반복이 많아요'),
    'sentiment'       : ('감성 점수',      '감성 표현이 편향됐어요'),
    'rating'          : ('별점',           '입력된 별점이에요'),
    'is_single_review': ('싱글 리뷰어',   '리뷰 이력이 없는 계정이에요'),
    'is_cold_start'   : ('콜드스타트',    '리뷰 수가 적은 식당이에요'),
    'weekday'         : ('요일',           '리뷰 작성 요일이에요'),
    'month'           : ('월',             '리뷰 작성 월이에요'),
    'digit_ratio'     : ('숫자 비율',      '숫자가 많이 포함됐어요'),
    'special_ratio'   : ('특수문자 비율',  '특수문자가 사용됐어요'),
    'menu_mentions'   : ('메뉴 언급',      '음식 메뉴 단어가 있어요'),
}

# ── 아티팩트 + 모델 로드 (셀 실행 시 1회) ───────────────────────────
_model, _tfidf, _svd, _scaler = None, None, None, None
_FEAT_COLS, _ALL_FEATS = [], []
_PROD_AVG, _PROD_COLD, _USER_BURST, _USER_SINGLE = {}, {}, {}, {}

try:
    from catboost import CatBoostClassifier, Pool as _Pool
    with open('dashboard_artifacts.pkl', 'rb') as _f:
        _arts = pickle.load(_f)
    _tfidf     = _arts['tfidf']
    _svd       = _arts['svd']
    _scaler    = _arts['scaler']
    _FEAT_COLS = _arts['feat_cols']
    _ALL_FEATS = _arts['all_feats']
    _PROD_AVG  = _arts['prod_avg_rating']
    _PROD_COLD = _arts['prod_is_cold']
    _USER_BURST  = _arts['user_burst']
    _USER_SINGLE = _arts['user_is_single']
    _model = CatBoostClassifier()
    _model.load_model('best_catboost_82.cbm')
    print(f"✅ 모델 로드 완료 | 피처 {len(_ALL_FEATS)}차원 | CatBoost 준비됨")
except Exception as _e:
    print(f"⚠️  모델 로드 실패 ({_e}) — 더미 모드로 동작합니다")


def _text_stats(text):
    words = text.lower().split()
    nw = max(len(words), 1)
    nc = max(len(text), 1)
    cnt = lambda kw: sum(1 for w in words if w in kw)
    fi = cnt(_FRAUD_KW) / nw
    ni = cnt(_NORMAL_KW) / nw
    return {
        'text_len': len(text), 'word_count': nw,
        'digit_ratio':   sum(c.isdigit() for c in text) / nc,
        'special_ratio': sum(1 for c in text if not c.isalnum() and not c.isspace()) / nc,
        'ttr':           len(set(words)) / nw,
        'sentiment':     (cnt(_POS_WORDS) - cnt(_NEG_WORDS)) / nw,
        'menu_mentions': cnt(_MENU_KW),
        'fraud_intensity':  fi,
        'normal_intensity': ni,
        'vocab_discrim':    fi - ni,
    }


def _build_html(score, shap_items, is_dummy=False):
    pct = score * 100
    if score >= 0.7:
        badge, bc, bg, gc = '사기 의심', '#FF4D4F', '#FFF1F0', '#FF4D4F'
    elif score >= 0.4:
        badge, bc, bg, gc = '주의 필요', '#FA8C16', '#FFF7E6', '#FA8C16'
    else:
        badge, bc, bg, gc = '정상',     '#52C41A', '#F6FFED', '#52C41A'

    rows = ''
    if shap_items:
        mx = max(abs(s['val']) for s in shap_items) or 1.0
        for s in shap_items:
            label, desc = SHAP_META.get(s['feat'], (s['feat'], ''))
            bar_c = '#3182F6' if s['val'] > 0 else '#52C41A'
            bw    = int(abs(s['val']) / mx * 100)
            vstr  = f"+{s['val']:.3f}" if s['val'] > 0 else f"{s['val']:.3f}"
            rows += (
                f'<div style="margin-bottom:18px">'
                f'<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:5px">'
                f'<span style="font-size:14px;font-weight:500;color:#191F28">{label}</span>'
                f'<span style="font-size:12px;font-weight:600;color:{bar_c}">{vstr}</span>'
                f'</div>'
                f'<div style="background:#F2F4F6;border-radius:4px;height:4px;margin-bottom:5px">'
                f'<div style="background:{bar_c};width:{bw}%;height:4px;border-radius:4px"></div>'
                f'</div>'
                f'<div style="font-size:12px;color:#6B7684;line-height:1.5">{desc}</div>'
                f'</div>'
            )

    dummy_note = (
        '<div style="font-size:11px;color:#aaa;margin-bottom:16px">'
        '※ 모델 미로드 — 텍스트 특성 기반 추정값</div>'
    ) if is_dummy else ''

    return (
        f'<div style="font-family:{_F}">'
        f'{dummy_note}'
        f'<div style="margin-bottom:24px">'
        f'<div style="font-size:13px;color:#6B7684;margin-bottom:8px">사기 확률</div>'
        f'<div style="display:flex;align-items:flex-end;gap:12px;margin-bottom:16px">'
        f'<span style="font-size:64px;font-weight:700;color:#191F28;line-height:1;letter-spacing:-2px">'
        f'{pct:.1f}'
        f'<span style="font-size:28px;font-weight:400;color:#6B7684">%</span>'
        f'</span>'
        f'<span style="font-size:14px;font-weight:600;color:{bc};background:{bg};'
        f'padding:6px 16px;border-radius:20px;margin-bottom:10px">{badge}</span>'
        f'</div>'
        f'<div style="background:#F2F4F6;border-radius:6px;height:6px">'
        f'<div style="background:{gc};width:{min(pct,100):.1f}%;height:6px;border-radius:6px"></div>'
        f'</div>'
        f'</div>'
        f'<div style="margin-top:24px;padding-top:20px;border-top:1px solid #F2F4F6">'
        f'<div style="font-size:14px;font-weight:600;color:#191F28;margin-bottom:18px">판정 근거</div>'
        f'{rows}'
        f'</div>'
        f'</div>'
    )


def predict_new_review(text, rating, user_id='', prod_id=''):
    time.sleep(0.6)
    text = str(text or '').strip()
    if not text:
        return (
            f'<div style="font-family:{_F};display:flex;align-items:center;'
            f'justify-content:center;min-height:280px;color:#B0B8C1;font-size:14px">'
            f'리뷰 텍스트를 입력하세요.</div>'
        )

    # ── 실제 모델 예측 ───────────────────────────────────────────────
    if _model is not None:
        try:
            rat    = float(rating or 3)
            now    = pd.Timestamp.now()
            pid    = int(str(prod_id).strip()) if str(prod_id).strip().isdigit() else None
            p_avg  = _PROD_AVG.get(pid, 3.5)
            p_cold = _PROD_COLD.get(pid, 0)
            r_dev  = abs(rat - p_avg)

            uid_s  = str(user_id or '').strip()
            uid_k  = int(uid_s) if uid_s.isdigit() else (uid_s if uid_s else None)
            burst  = _USER_BURST.get(uid_k, {'user_burst_max7d': 0, 'user_burst_ratio': 0})
            single = _USER_SINGLE.get(uid_k, 1)

            ts  = _text_stats(text)
            num = np.array([[
                rat, now.weekday(), now.month,
                r_dev, p_cold, single,
                burst['user_burst_max7d'], burst['user_burst_ratio'],
                ts['text_len'],    ts['word_count'],   ts['digit_ratio'],
                ts['special_ratio'], ts['ttr'],         ts['sentiment'],
                ts['menu_mentions'], ts['fraud_intensity'],
                ts['normal_intensity'], ts['vocab_discrim'],
            ]], dtype=np.float32)

            tv  = _tfidf.transform([text])
            sv  = _svd.transform(tv).astype(np.float32)
            x   = np.hstack([_scaler.transform(num), sv])
            score = float(_model.predict_proba(x)[0][1])

            # SHAP
            shap_items = None
            try:
                sv_arr = _model.get_feature_importance(
                    _Pool(x, feature_names=_ALL_FEATS), type='ShapValues'
                )[0][:-1]
                df_s = pd.DataFrame({'feat': _ALL_FEATS, 'val': sv_arr})
                top5 = df_s.reindex(df_s['val'].abs().nlargest(5).index).sort_values('val', ascending=False)
                shap_items = top5.to_dict('records')
            except Exception:
                pass

            return _build_html(score, shap_items, is_dummy=False)

        except Exception as e:
            return (
                f'<div style="font-family:{_F};padding:20px;color:{chr(35)}FF4D4F">'
                f'예측 오류: {e}</div>'
            )

    # ── 더미 폴백 (모델 없을 때) ─────────────────────────────────────
    h     = int(hashlib.md5(text.encode('utf-8', errors='ignore')).hexdigest(), 16)
    score = 0.05 + (h % 900) / 1000.0
    shap_items = [
        {'feat': 'fraud_intensity',  'val':  round(0.05 + (h % 100) / 800, 3)},
        {'feat': 'user_burst_ratio', 'val':  round(0.03 + (h % 60)  / 900, 3)},
        {'feat': 'text_len',         'val': -round(0.02 + (h % 40)  / 950, 3)},
        {'feat': 'rating_deviation', 'val':  round(0.02 + (h % 50)  / 1000, 3)},
        {'feat': 'sentiment',        'val': -round(0.01 + (h % 30)  / 1100, 3)},
    ]
    return _build_html(score, shap_items, is_dummy=True)


✅ 모델 로드 완료 | 피처 82차원 | CatBoost 준비됨


In [12]:

import gradio as gr
import pandas as pd, numpy as np, pickle, os, warnings
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

warnings.filterwarnings('ignore')
os.chdir(os.path.dirname(os.path.abspath('app.py')))

# ── 데이터 & 모델 로드 ───────────────────────────────────────────────
df_pre = pd.read_parquet('precomputed_dash.parquet')
df_pre['date'] = pd.to_datetime(df_pre['date'])

from catboost import CatBoostClassifier, Pool as _Pool
with open('dashboard_artifacts.pkl', 'rb') as f:
    arts = pickle.load(f)
tfidf, svd, scaler   = arts['tfidf'], arts['svd'], arts['scaler']
FEAT_COLS, ALL_FEATS = arts['feat_cols'], arts['all_feats']
PROD_AVG   = arts['prod_avg_rating']
PROD_COLD  = arts['prod_is_cold']
USER_BURST = arts['user_burst']
USER_SINGLE= arts['user_is_single']
_model = CatBoostClassifier()
_model.load_model('best_catboost_82.cbm')
print(f"✅ 모델 로드: {len(ALL_FEATS)}차원")

# ── 키워드 & SHAP 메타 ───────────────────────────────────────────────
_FRAUD_KW  = {'told','said','asked','manager','table','owner','call','rude','experience','server','waitress','waiter','horrible','worst','complained','never','disgusting','terrible','awful'}
_NORMAL_KW = {'topped','texture','creamy','sweetness','pieces','rich','tender','crispy','fluffy','flavor','delicious','fresh','tasty','amazing','fantastic','loved','enjoyed'}
_MENU_KW   = {'food','steak','burger','pizza','pasta','chicken','fish','salad','sandwich','soup','noodle','rice','sauce','cheese','beef','pork','bread','dessert','cake','coffee','drink'}
_POS = {'good','great','excellent','amazing','wonderful','love','best','perfect','fantastic','delicious','fresh','friendly','enjoyed','recommend','awesome'}
_NEG = {'bad','terrible','awful','horrible','worst','rude','slow','dirty','disgusting','disappointing','never','poor'}

_SVD_DESC = '리뷰의 문체와 어휘 패턴이에요'
SHAP_META = {f'SVD_{i}': (f'텍스트 스타일 ({i+1})', _SVD_DESC) for i in range(64)}
SHAP_META.update({
    'rating':           ('별점',              '입력된 별점이에요'),
    'weekday':          ('작성 요일',          '리뷰 작성 요일이에요'),
    'month':            ('작성 월',            '리뷰 작성 월이에요'),
    'rating_deviation': ('평균과 동떨어진 별점','해당 식당 평균과 얼마나 다른지예요'),
    'is_cold_start':    ('신규 유저 여부',     '리뷰가 적은 새 계정이에요'),
    'is_single_review': ('신규 유저 여부',     '리뷰가 적은 새 계정이에요'),
    'user_burst_max7d': ('7일내 최대 리뷰 수', '짧은 기간에 몰아쓴 리뷰 수예요'),
    'user_burst_ratio': ('버스트 집중도',      '전체 리뷰 중 집중 작성 비율이에요'),
    'text_len':         ('리뷰 길이',          '리뷰 텍스트의 글자 수예요'),
    'word_count':       ('단어 수',            '리뷰에 사용된 단어 수예요'),
    'digit_ratio':      ('숫자 비율',          '리뷰 내 숫자 포함 비율이에요'),
    'special_ratio':    ('특수문자 비율',       '특수문자 사용 비율이에요'),
    'ttr':              ('어휘 다양성',         '단어 반복이 많을수록 낮아져요'),
    'sentiment':        ('감성 점수',           '긍정·부정 어휘의 균형이에요'),
    'menu_mentions':    ('메뉴 언급 수',        '음식 메뉴 관련 단어 수예요'),
    'fraud_intensity':  ('사기 어휘 비율',      '사기 리뷰에서 자주 쓰는 단어 비율이에요'),
    'normal_intensity': ('정상 어휘 비율',      '정상 리뷰에서 자주 쓰는 단어 비율이에요'),
    'vocab_discrim':    ('어휘 변별 점수',       '사기·정상 어휘 비율의 차이예요'),
    'norm_rating_entropy':('별점 다양성',       '항상 같은 별점만 주면 낮아져요'),
})

def _ts(text):
    words = text.lower().split()
    nw, nc = max(len(words),1), max(len(text),1)
    cnt = lambda kw: sum(1 for w in words if w in kw)
    fi, ni = cnt(_FRAUD_KW)/nw, cnt(_NORMAL_KW)/nw
    return {'text_len':len(text),'word_count':nw,
            'digit_ratio':sum(c.isdigit() for c in text)/nc,
            'special_ratio':sum(1 for c in text if not c.isalnum() and not c.isspace())/nc,
            'ttr':len(set(words))/nw,'sentiment':(cnt(_POS)-cnt(_NEG))/nw,
            'menu_mentions':cnt(_MENU_KW),'fraud_intensity':fi,
            'normal_intensity':ni,'vocab_discrim':fi-ni}

# ── Gradio: / 에서 /app 으로 자동 리다이렉트 ────────────────────────
with gr.Blocks(title="사기 리뷰 탐지 대시보드") as demo:
    gr.HTML("""
    <script>window.location.replace("/app");</script>
    <noscript>
      <meta http-equiv="refresh" content="0;url=/app">
    </noscript>
    """)

_, local_url, share_url = demo.launch(
    share=True,
    prevent_thread_lock=True,
    quiet=True,
)
app = demo.server_app

# ── 대시보드 HTML (/app) ─────────────────────────────────────────────
@app.get("/app")
async def dashboard():
    return HTMLResponse(open('dashboard.html', encoding='utf-8').read())

# ── API 라우트 ───────────────────────────────────────────────────────
@app.get("/api/overview")
async def api_overview():
    total      = len(df_pre)
    fraud_pred = int(df_pre['fraud_pred'].sum())
    fraud_rate = round(fraud_pred / total * 100, 1)
    tmp = df_pre.copy()
    tmp['ym'] = tmp['date'].dt.to_period('M').astype(str)
    g = tmp.groupby('ym').agg(total_=('label','count'),fp=('fraud_pred','sum'),fa=('label','sum')).reset_index()
    trend = {'labels':g['ym'].tolist(),
             'pred':(g['fp']/g['total_']*100).round(1).tolist(),
             'actual':(g['fa']/g['total_']*100).round(1).tolist()}
    rg = df_pre.groupby('prod_id').agg(total_=('fraud_score','count'),fraud=('fraud_pred','sum')).reset_index()
    rg['rate'] = (rg['fraud']/rg['total_']*100).round(1)
    top_r = rg.nlargest(10,'rate').sort_values('rate')
    rest_chart = {'labels':['#'+str(x) for x in top_r['prod_id']],'values':top_r['rate'].tolist()}
    ug = df_pre.groupby('user_id').agg(cnt=('fraud_score','count'),avg=('fraud_score','mean')).reset_index()
    top_u = ug[ug['cnt']>=3].nlargest(10,'avg').sort_values('avg')
    user_chart = {'labels':['#'+str(x) for x in top_u['user_id']],'values':top_u['avg'].round(3).tolist()}
    best = rg.nlargest(1,'rate').iloc[0]
    return {'metrics':{'total':total,'fraud_pred':fraud_pred,'fraud_rate':fraud_rate,
                       'top_restaurant':f"#{int(best['prod_id'])}",
                       'top_restaurant_rate':float(best['rate'])},
            'trend':trend,'rest_chart':rest_chart,'user_chart':user_chart}

class PReq(BaseModel):
    text: str; rating: float=3.0; user_id: str=''; prod_id: str=''

@app.post("/api/predict")
async def api_predict(req: PReq):
    text = req.text.strip()
    if not text:
        return {"error":"텍스트를 입력하세요"}
    try:
        rat   = float(req.rating or 3)
        now   = pd.Timestamp.now()
        pid   = int(req.prod_id) if req.prod_id.strip().isdigit() else None
        p_avg = PROD_AVG.get(pid, 3.5); p_cold = PROD_COLD.get(pid, 0)
        uid_s = req.user_id.strip()
        uid_k = int(uid_s) if uid_s.isdigit() else (uid_s if uid_s else None)
        burst = USER_BURST.get(uid_k, {'user_burst_max7d':0,'user_burst_ratio':0})
        ts = _ts(text)
        num = np.array([[rat, now.weekday(), now.month, abs(rat-p_avg), p_cold,
                         USER_SINGLE.get(uid_k,1), burst['user_burst_max7d'],
                         burst['user_burst_ratio'], ts['text_len'], ts['word_count'],
                         ts['digit_ratio'], ts['special_ratio'], ts['ttr'],
                         ts['sentiment'], ts['menu_mentions'], ts['fraud_intensity'],
                         ts['normal_intensity'], ts['vocab_discrim']]], dtype=np.float32)
        tv    = tfidf.transform([text])
        sv    = svd.transform(tv).astype(np.float32)
        x     = np.hstack([scaler.transform(num), sv])
        score = float(_model.predict_proba(x)[0][1])
        sv_arr= _model.get_feature_importance(_Pool(x, feature_names=ALL_FEATS), type='ShapValues')[0][:-1]
        feat_df = pd.DataFrame({'feat':ALL_FEATS,'val':sv_arr})
        top5 = feat_df.reindex(feat_df['val'].abs().nlargest(5).index).sort_values('val',ascending=False)
        shap = [{'label':SHAP_META.get(r.feat,(r.feat,''))[0],
                 'desc':SHAP_META.get(r.feat,('',''))[1],
                 'val':round(float(r.val),4)} for _,r in top5.iterrows()]
        return {'score':round(score,4),'pct':round(score*100,1),'shap':shap,'dummy':False}
    except Exception as e:
        return {"error":str(e)}

@app.get("/api/restaurant/{pid}")
async def api_restaurant(pid: int):
    sub = df_pre[df_pre['prod_id']==pid]
    if sub.empty: return {"error":f"식당 {pid} 데이터 없음"}
    total=len(sub); fraud_cnt=int(sub['fraud_pred'].sum())
    fraud_rate=round(fraud_cnt/total*100,1); avg_score=round(sub['fraud_score'].mean()*100,1)
    users=int(sub['user_id'].nunique())
    period=f"{sub['date'].min().strftime('%Y-%m-%d')} ~ {sub['date'].max().strftime('%Y-%m-%d')}"
    bins=np.linspace(0,1,21); labels,nc,fc=[],[],[]
    for i in range(len(bins)-1):
        mask=(sub['fraud_score']>=bins[i])&(sub['fraud_score']<bins[i+1])
        labels.append(f"{bins[i]:.1f}")
        nc.append(int(sub[mask&(sub['fraud_pred']==0)].shape[0]))
        fc.append(int(sub[mask&(sub['fraud_pred']==1)].shape[0]))
    disp=sub[['date','user_id','rating','fraud_score','text']].copy()
    disp=disp.sort_values('fraud_score',ascending=False).head(20)
    disp['date']=disp['date'].dt.strftime('%Y-%m-%d')
    disp['fraud_score']=disp['fraud_score'].round(4)
    disp['badge']=disp['fraud_score'].apply(lambda s:'사기 의심' if s>=0.7 else('주의' if s>=0.4 else '정상'))
    disp['text']=disp['text'].str[:80]
    return {'metrics':{'total':total,'users':users,'fraud_rate':fraud_rate,'avg_score':avg_score,'period':period},
            'dist':{'labels':labels,'normal':nc,'fraud':fc},
            'reviews':disp[['date','user_id','rating','fraud_score','badge','text']].to_dict('records')}

@app.get("/api/user/{uid}")
async def api_user(uid: int):
    sub = df_pre[df_pre['user_id']==uid]
    if sub.empty: return {"error":f"유저 {uid} 데이터 없음"}
    total=len(sub); fraud_cnt=int(sub['fraud_pred'].sum())
    fraud_rate=round(fraud_cnt/total*100,1); avg_score=round(float(sub['fraud_score'].mean()),4)
    restaurants=int(sub['prod_id'].nunique())
    span=int((sub['date'].max()-sub['date'].min()).days)
    sub_s=sub.sort_values('date').copy()
    sub_s['gap']=sub_s['date'].diff().dt.days.fillna(999)
    burst=bool((sub_s['gap']<3).sum()>1)
    sub_s['date_str']=sub_s['date'].dt.strftime('%Y-%m-%d')
    timeline=[{'x':r.date_str,'y':round(r.fraud_score,4),'pred':int(r.fraud_pred),'text':str(r.text)[:60]}
              for r in sub_s[['date_str','fraud_score','fraud_pred','text']].itertuples()]
    disp=sub[['date','prod_id','rating','fraud_score','text']].copy()
    disp=disp.sort_values('fraud_score',ascending=False)
    disp['date']=disp['date'].dt.strftime('%Y-%m-%d')
    disp['fraud_score']=disp['fraud_score'].round(4)
    disp['badge']=disp['fraud_score'].apply(lambda s:'사기 의심' if s>=0.7 else('주의' if s>=0.4 else '정상'))
    disp['text']=disp['text'].str[:80]
    return {'metrics':{'total':total,'restaurants':restaurants,'fraud_rate':fraud_rate,
                       'avg_score':avg_score,'span':span,'burst':burst},
            'timeline':timeline,
            'reviews':disp[['date','prod_id','rating','fraud_score','badge','text']].to_dict('records')}

print("\n" + "="*60)
print(f"  공개 URL : {share_url}/app   (1주일 유효)")
print(f"  로컬 URL : {local_url}app")
print("="*60)
print("  노트북이 실행 중인 동안 위 링크로 접속 가능합니다.")


✅ 모델 로드: 82차원
* Running on public URL: https://c391cee2b9c4edbaed.gradio.live



  공개 URL : https://c391cee2b9c4edbaed.gradio.live/app   (1주일 유효)
  로컬 URL : http://127.0.0.1:7860/app
  노트북이 실행 중인 동안 위 링크로 접속 가능합니다.
